# Composer Forward Speed Benchmark

Compare wall-clock speed of:
1. Original `ActionComposer.encode_sequence_path` / `encode_target_path` (has `.any()` CPU-GPU syncs)
2. Vectorized variants (unconditional compute + mask)
3. Original under `torch.profiler` to confirm where time is spent

Run this in parallel with any ongoing training. ~2 min per benchmark test.

## Setup

In [1]:
import torch
import torch.nn.functional as F
import random
import time
import sys
from pathlib import Path
import numpy as np
import types

sys.path.insert(0, str(Path.cwd().parent))

from biojepa_v0_7 import ActionComposer, ActionComposerConfig
from training_v0_7 import load_feature_banks, get_seq_embeddings, get_target_embeddings, reset_seed
from dataloader_v0_7 import ComposerLoader
from config_v0_7 import DataConfig

torch.manual_seed(1337)
random.seed(1337)
torch.set_float32_matmul_precision('high')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

data_root = Path('~/data/jepa/v0_7').expanduser()
ref_root = Path('~/data/jepa/reference_data').expanduser()
data_cfg = DataConfig(data_root=data_root, checkpoint_dir=data_root / 'checkpoints', ref_dir=ref_root)

seq_banks, target_bank = load_feature_banks(data_cfg, device)
batch_size = 64
chemical_fraction = 0.1
pert_dir = data_root / 'pert_embd'


KeyboardInterrupt: 

## Benchmark harness

In [ ]:
def run_benchmark(composer, n_steps=500, warmup=50, temperature=0.00126, label=""):
    """Measure wall-clock time per step for composer forward+backward+step."""
    reset_seed(1337)
    train_loader = ComposerLoader(batch_size, 'train', pert_dir, device, seed=1337, chemical_fraction=chemical_fraction)
    optimizer = torch.optim.AdamW(composer.parameters(), lr=3.6e-4, weight_decay=0.003, fused=True)
    composer.train()

    # Warmup
    for _ in range(warmup):
        b = train_loader.next_batch()
        B = b.seq_idx.shape[0]
        seq_emb = get_seq_embeddings(b.seq_idx.unsqueeze(1), b.modality.unsqueeze(1), seq_banks)
        target_emb = get_target_embeddings(b.target_idx.unsqueeze(1), target_bank)
        mode_ids = b.mode.unsqueeze(1); modality_ids = b.modality.unsqueeze(1)
        pert_mask = torch.ones(B, 1, dtype=torch.bool, device=device)
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=True):
            z_seq = composer.encode_sequence_path(seq_emb, modality_ids, mode_ids, pert_mask).squeeze(1)
            z_target = composer.encode_target_path(target_emb, mode_ids, pert_mask).squeeze(1)
            z_seq = F.normalize(z_seq, dim=1); z_target = F.normalize(z_target, dim=1)
            logits = torch.matmul(z_seq, z_target.T) / temperature
            loss = F.cross_entropy(logits, torch.arange(B, device=device))
        loss.backward(); optimizer.step()

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    losses = []
    for _ in range(n_steps):
        b = train_loader.next_batch()
        B = b.seq_idx.shape[0]
        seq_emb = get_seq_embeddings(b.seq_idx.unsqueeze(1), b.modality.unsqueeze(1), seq_banks)
        target_emb = get_target_embeddings(b.target_idx.unsqueeze(1), target_bank)
        mode_ids = b.mode.unsqueeze(1); modality_ids = b.modality.unsqueeze(1)
        pert_mask = torch.ones(B, 1, dtype=torch.bool, device=device)
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=True):
            z_seq = composer.encode_sequence_path(seq_emb, modality_ids, mode_ids, pert_mask).squeeze(1)
            z_target = composer.encode_target_path(target_emb, mode_ids, pert_mask).squeeze(1)
            z_seq = F.normalize(z_seq, dim=1); z_target = F.normalize(z_target, dim=1)
            logits = torch.matmul(z_seq, z_target.T) / temperature
            loss = F.cross_entropy(logits, torch.arange(B, device=device))
        loss.backward(); optimizer.step()
        losses.append(loss.item())
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    print(f'{label:30s} | {n_steps} steps in {elapsed:.2f}s | {elapsed/n_steps*1000:.2f} ms/step | last loss {losses[-1]:.4f}')
    return {'label': label, 'elapsed': elapsed, 'n_steps': n_steps, 'ms_per_step': elapsed/n_steps*1000, 'losses': losses}


## Test 1: Original composer

Baseline. Has `.any()` CPU-GPU syncs in the modality loop.

In [ ]:
reset_seed(1337)
cfg = ActionComposerConfig(latent_dim=128, mode_dim=64, heads=4)
composer_orig = ActionComposer(cfg).to(device)
result_orig = run_benchmark(composer_orig, n_steps=500, label='original')


## Test 2: Vectorized composer

Monkey-patch `encode_sequence_path` and `encode_target_path` to run projections unconditionally
and mask outputs instead of using boolean indexing + `.any()`. Mathematically equivalent output,
no CPU-GPU syncs in the forward.

In [ ]:
def encode_sequence_path_vec(self, seq_emb, modality_ids, mode_ids, pert_mask):
    B, N_pert = modality_ids.shape
    D = self.config.latent_dim
    device = modality_ids.device
    action_latents = torch.zeros(B, N_pert, D, device=device)
    for p in range(N_pert):
        p_mask = pert_mask[:, p]
        p_modality = modality_ids[:, p]
        p_mode = mode_ids[:, p]
        seq_lat = torch.zeros(B, D, device=device)
        for mod_id in range(3):
            proj = self.seq_projectors[self.modality_to_key[mod_id]]
            mod_mask = ((p_modality == mod_id) & p_mask).unsqueeze(-1).to(seq_emb.dtype)
            seq_lat = seq_lat + proj(seq_emb[:, p, :proj.in_features]) * mod_mask
        action = self._apply_mode(seq_lat, p_mode)
        action_latents[:, p] = action * p_mask.to(action.dtype).unsqueeze(-1)
    return action_latents

def encode_target_path_vec(self, target_emb, mode_ids, pert_mask):
    B, N_pert = mode_ids.shape
    D = self.config.latent_dim
    device = mode_ids.device
    action_latents = torch.zeros(B, N_pert, D, device=device)
    for p in range(N_pert):
        p_mask = pert_mask[:, p].unsqueeze(-1).to(target_emb.dtype)
        target_lat = self.target_projector(target_emb[:, p]) * p_mask
        action = self._apply_mode(target_lat, mode_ids[:, p])
        action_latents[:, p] = action * p_mask
    return action_latents

reset_seed(1337)
composer_vec = ActionComposer(cfg).to(device)
composer_vec.encode_sequence_path = types.MethodType(encode_sequence_path_vec, composer_vec)
composer_vec.encode_target_path = types.MethodType(encode_target_path_vec, composer_vec)
result_vec = run_benchmark(composer_vec, n_steps=500, label='vectorized')


## Test 3: Profile original

Confirm where the time is going. Top 20 ops by CUDA time.

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

reset_seed(1337)
composer_prof = ActionComposer(cfg).to(device)
train_loader = ComposerLoader(batch_size, 'train', pert_dir, device, seed=1337, chemical_fraction=chemical_fraction)
optimizer = torch.optim.AdamW(composer_prof.parameters(), lr=3.6e-4, weight_decay=0.003, fused=True)
composer_prof.train()

# warmup
for _ in range(20):
    b = train_loader.next_batch()
    B = b.seq_idx.shape[0]
    seq_emb = get_seq_embeddings(b.seq_idx.unsqueeze(1), b.modality.unsqueeze(1), seq_banks)
    target_emb = get_target_embeddings(b.target_idx.unsqueeze(1), target_bank)
    mode_ids = b.mode.unsqueeze(1); modality_ids = b.modality.unsqueeze(1)
    pert_mask = torch.ones(B, 1, dtype=torch.bool, device=device)
    optimizer.zero_grad()
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=True):
        z_seq = composer_prof.encode_sequence_path(seq_emb, modality_ids, mode_ids, pert_mask).squeeze(1)
        z_target = composer_prof.encode_target_path(target_emb, mode_ids, pert_mask).squeeze(1)
        z_seq = F.normalize(z_seq, dim=1); z_target = F.normalize(z_target, dim=1)
        logits = torch.matmul(z_seq, z_target.T) / 0.00126
        loss = F.cross_entropy(logits, torch.arange(B, device=device))
    loss.backward(); optimizer.step()

torch.cuda.synchronize()

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=False) as prof:
    for _ in range(30):
        b = train_loader.next_batch()
        B = b.seq_idx.shape[0]
        seq_emb = get_seq_embeddings(b.seq_idx.unsqueeze(1), b.modality.unsqueeze(1), seq_banks)
        target_emb = get_target_embeddings(b.target_idx.unsqueeze(1), target_bank)
        mode_ids = b.mode.unsqueeze(1); modality_ids = b.modality.unsqueeze(1)
        pert_mask = torch.ones(B, 1, dtype=torch.bool, device=device)
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=True):
            z_seq = composer_prof.encode_sequence_path(seq_emb, modality_ids, mode_ids, pert_mask).squeeze(1)
            z_target = composer_prof.encode_target_path(target_emb, mode_ids, pert_mask).squeeze(1)
            z_seq = F.normalize(z_seq, dim=1); z_target = F.normalize(z_target, dim=1)
            logits = torch.matmul(z_seq, z_target.T) / 0.00126
            loss = F.cross_entropy(logits, torch.arange(B, device=device))
        loss.backward(); optimizer.step()
    torch.cuda.synchronize()

print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=20))
print()
print(prof.key_averages().table(sort_by='cpu_time_total', row_limit=20))


## Numerical equivalence check

Confirm vectorized version produces ~same outputs as original on same input.

In [ ]:
reset_seed(1337)
composer_a = ActionComposer(cfg).to(device)
reset_seed(1337)
composer_b = ActionComposer(cfg).to(device)
composer_b.encode_sequence_path = types.MethodType(encode_sequence_path_vec, composer_b)
composer_b.encode_target_path = types.MethodType(encode_target_path_vec, composer_b)

train_loader = ComposerLoader(batch_size, 'train', pert_dir, device, seed=1337, chemical_fraction=chemical_fraction)
b = train_loader.next_batch()
B = b.seq_idx.shape[0]
seq_emb = get_seq_embeddings(b.seq_idx.unsqueeze(1), b.modality.unsqueeze(1), seq_banks)
target_emb = get_target_embeddings(b.target_idx.unsqueeze(1), target_bank)
mode_ids = b.mode.unsqueeze(1); modality_ids = b.modality.unsqueeze(1)
pert_mask = torch.ones(B, 1, dtype=torch.bool, device=device)

with torch.no_grad():
    za_seq = composer_a.encode_sequence_path(seq_emb, modality_ids, mode_ids, pert_mask)
    zb_seq = composer_b.encode_sequence_path(seq_emb, modality_ids, mode_ids, pert_mask)
    za_tgt = composer_a.encode_target_path(target_emb, mode_ids, pert_mask)
    zb_tgt = composer_b.encode_target_path(target_emb, mode_ids, pert_mask)

print(f'seq_path max diff: {(za_seq - zb_seq).abs().max().item():.2e}')
print(f'seq_path mean diff: {(za_seq - zb_seq).abs().mean().item():.2e}')
print(f'target_path max diff: {(za_tgt - zb_tgt).abs().max().item():.2e}')
print(f'target_path mean diff: {(za_tgt - zb_tgt).abs().mean().item():.2e}')


## Summary

In [ ]:
print(f'original:   {result_orig["ms_per_step"]:.2f} ms/step')
print(f'vectorized: {result_vec["ms_per_step"]:.2f} ms/step')
speedup = result_orig['ms_per_step'] / result_vec['ms_per_step']
print(f'speedup:    {speedup:.2f}x')
print()
est_full_orig = result_orig['ms_per_step'] * 1_840_000 / 1000 / 3600
est_full_vec = result_vec['ms_per_step'] * 1_840_000 / 1000 / 3600
print(f'full 1.84M-step run estimate:')
print(f'  original:   {est_full_orig:.1f} hrs')
print(f'  vectorized: {est_full_vec:.1f} hrs')
